In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')
FOLDERNAME = 'cs231n/project/'
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

In [ ]:
# Check GPU availability

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# Check TensorFlow and PyTorch versions
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("TensorFlow version: ", tf.__version__)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Calculate PSNR, SSIM, and LPIPS

import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import os
from PIL import Image
from scipy.stats import norm
import cv2

def calculate_metrics(ground_truth, predicted):
    psnr_value = psnr(ground_truth, predicted)
    ssim_value = ssim(ground_truth, predicted, multichannel=True)

    # Initialize the LPIPS model
    lpips_model = lpips.LPIPS(net='alex')
    lpips_value = lpips_model(ground_truth, predicted).item()

    return psnr_value, ssim_value, lpips_value

# Folder paths for ground truth and predicted images
ground_truth_folder = '/content/drive/My Drive/tmp/ground_truth'
predicted_folder = '/content/drive/My Drive/tmp/predicted'

# Get the list of image files in the ground truth folder
ground_truth_files = [f for f in os.listdir(ground_truth_folder) if f.endswith(".png") or f.endswith(".jpg")]

for gt_file in ground_truth_files:
    ground_truth_path = os.path.join(ground_truth_folder, gt_file)
    predicted_path = os.path.join(predicted_folder, gt_file)  # Assuming the predicted image has the same filename

    ground_truth = Image.open(ground_truth_path)
    predicted = Image.open(predicted_path)

    psnr_score, ssim_score, lpips_score = calculate_metrics(ground_truth, predicted)
    print(f"Image: {gt_file}")
    print(f"  PSNR: {psnr_score:.2f}")
    print(f"  SSIM: {ssim_score:.2f}")
    print(f"  LPIPS: {lpips_score:.4f}")

Mounted at /content/drive


In [ ]:
# Compare with NeRF, InstantNGP, and Plenoxels

# - ground_truth_folder
#   - image1.png
#   - image2.png
#   - ...
# - gaussian_splatting_folder
#   - image1.png
#   - image2.png
#   - ...
# - nerf_folder
#   - image1.png
#   - image2.png
#   - ...
# - instantngp_folder
#   - image1.png
#   - image2.png
#   - ...
# - plenoxels_folder
#   - image1.png
#   - image2.png
#   - ...

# Folder paths for ground truth and predicted images
ground_truth_folder = "/content/drive/My Drive/tmp/ground_truth_folder"
gaussian_splatting_folder = "/content/drive/My Drive/tmp/gaussian_splatting_folder"
nerf_folder = "/content/drive/My Drive/tmp/nerf_folder"
instantngp_folder = "/content/drive/My Drive/tmp/instantngp_folder"
plenoxels_folder = "/content/drive/My Drive/tmp/plenoxels_folder"

# Get the list of image files in the ground truth folder
ground_truth_files = [f for f in os.listdir(ground_truth_folder) if f.endswith(".png") or f.endswith(".jpg")]

methods = [
    ("Gaussian Splatting", gaussian_splatting_folder),
    ("NeRF", nerf_folder),
    ("InstantNGP", instantngp_folder),
    ("Plenoxels", plenoxels_folder)
]

for method_name, predicted_folder in methods:
    print(f"Method: {method_name}")
    metrics = []

    for gt_file in ground_truth_files:
        ground_truth_path = os.path.join(ground_truth_folder, gt_file)
        predicted_path = os.path.join(predicted_folder, gt_file)

        ground_truth = Image.open(ground_truth_path)
        predicted = Image.open(predicted_path)

        psnr_score, ssim_score, lpips_score = calculate_metrics(ground_truth, predicted)
        metrics.append((psnr_score, ssim_score, lpips_score))

    avg_psnr = sum([m[0] for m in metrics]) / len(metrics)
    avg_ssim = sum([m[1] for m in metrics]) / len(metrics)
    avg_lpips = sum([m[2] for m in metrics]) / len(metrics)

    print(f"  Average PSNR: {avg_psnr:.2f}")
    print(f"  Average SSIM: {avg_ssim:.2f}")
    print(f"  Average LPIPS: {avg_lpips:.4f}")

In [ ]:
# Ablation Studies - modifying Gaussian parameters

# - ground_truth_folder
#   - image1.png
#   - image2.png
#   - ...
# - gaussian_splatting_folder_1000
#   - image1.png
#   - image2.png
#   - ...
# - gaussian_splatting_folder_5000
#   - image1.png
#   - image2.png
#   - ...
# - gaussian_splatting_folder_10000
#   - image1.png
#   - image2.png
#   - ...

# Folder paths for ground truth and predicted images
ground_truth_folder = "path/to/ground_truth_folder"
gaussian_splatting_folders = [
    ("1000 Gaussians", "/content/drive/My Drive/tmp/gaussian_splatting_folder_1000"),
    ("5000 Gaussians", "/content/drive/My Drive/tmp/gaussian_splatting_folder_5000"),
    ("10000 Gaussians", "/content/drive/My Drive/tmp/gaussian_splatting_folder_10000")
]

gaussian_params = [
    norm(loc=0, scale=1),  # Gaussian with mean=0, std=1
    norm(loc=0, scale=2),  # Gaussian with mean=0, std=2
    norm(loc=1, scale=1)   # Gaussian with mean=1, std=1
]

for params in gaussian_params:
    print(f"Configuration: {params}")
    metrics = []

    for gt_file in ground_truth_files:
        ground_truth_path = os.path.join(ground_truth_folder, gt_file)
        predicted_path = os.path.join(gaussian_splatting_folder, gt_file)

        ground_truth = cv2.imread(ground_truth_path)
        predicted = cv2.imread(predicted_path)

        # Apply Gaussian noise to the predicted image
        noise = params.rvs(size=predicted.shape)
        predicted_noisy = predicted + noise

        psnr_score, ssim_score, lpips_score = calculate_metrics(ground_truth, predicted_noisy)
        metrics.append((psnr_score, ssim_score, lpips_score))

# Get the list of image files in the ground truth folder
ground_truth_files = [f for f in os.listdir(ground_truth_folder) if f.endswith(".png") or f.endswith(".jpg")]

for config_name, predicted_folder in gaussian_splatting_folders:
    print(f"Configuration: {config_name}")
    metrics = []

    for gt_file in ground_truth_files:
        ground_truth_path = os.path.join(ground_truth_folder, gt_file)
        predicted_path = os.path.join(predicted_folder, gt_file)

        ground_truth = Image.open(ground_truth_path)
        predicted = Image.open(predicted_path)

        psnr_score, ssim_score, lpips_score = calculate_metrics(ground_truth, predicted)
        metrics.append((psnr_score, ssim_score, lpips_score))

    avg_psnr = sum([m[0] for m in metrics]) / len(metrics)
    avg_ssim = sum([m[1] for m in metrics]) / len(metrics)
    avg_lpips = sum([m[2] for m in metrics]) / len(metrics)

    print(f"  Average PSNR: {avg_psnr:.2f}")
    print(f"  Average SSIM: {avg_ssim:.2f}")
    print(f"  Average LPIPS: {avg_lpips:.4f}")